# 03. 출시 후 LLM 리뷰 분석

`01_preprocess_reviews.ipynb`전처리 파일에서 만든 후보 데이터를 받아서\
분석 조건 필터링과 샘플링을 적용한 뒤 **PydanticAI + Vertex AI 기반 Gemini**로 Steam 리뷰 감성·이슈 분석을 실행한다.


## 역할
1. 전처리 완료 후보 리뷰를 불러온다.
2. 특정 게임, 최근 리뷰 구간, 언어, 리뷰 신뢰도 조건을 적용한다.
3. 리뷰 수를 제한하고 부정 리뷰를 우선 확보하는 방식으로 샘플링한다.
4. LLM 입력 파일을 저장한다.
5. Vertex AI 기반 Gemini 모델로 리뷰별 감정과 이슈를 분석한다.
6. 보고서에서 사용할 CSV 산출물을 저장한다.

## 이 코드에서 사용하는 주요 입력
- `llm_preprocessed_reviews.csv`
- `llm_candidate_game_summary.csv`

## 이 코드에서 생성하는 주요 산출물
- `llm_input_reviews.csv`
- `llm_review_analysis_result.csv`
- `llm_issue_tags_flat.csv`

## 분석 흐름

```text
특정 게임 리뷰 데이터
        ↓
최근 리뷰 구간 필터링
        ↓
영어 리뷰 / Steam 구매 / 무료 수령 제외 / 얼리액세스 제외
        ↓
부정 리뷰 우선 샘플링
        ↓
LLM 입력 파일 저장
llm_input_reviews.csv
        ↓
Vertex AI Gemini 호출
PydanticAI 구조화 출력
        ↓
리뷰 단위 결과 저장
llm_review_analysis_result.csv
        ↓
이슈 태그 펼치기
llm_issue_tags_flat.csv
        ↓
03-1에서 패치·운영 근거표 생성
```


# 0. 환경설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import ast
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

# ============================================================
# 데이터 분석용 라이브러리
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ============================================================
# 환경변수 / 진행률 / LLM 출력 스키마 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider



# ============================================================
# 한글 폰트 설정
# ============================================================
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:  # Linux
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 경로 설정

In [2]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 출시 후 분석 대상 게임 설정
# ============================================================
# 여러 게임을 한 번에 실행하려면 RUN_MULTI_GAME_MODE=True로 두고,
# TARGET_GAME_KEYS에 실행할 게임 key를 넣으면 된다.
# 게임 1개만 실행하려면 RUN_MULTI_GAME_MODE=False로 바꾸고 TARGET_GAME_KEY만 변경한다.

RUN_MULTI_GAME_MODE = True

POSTLAUNCH_TARGET_GAMES = {
    "heroes_of_hammerwatch_2": {
        "appid": 619820,
        "game_name": "Heroes of Hammerwatch II",
        "game_slug": "heroes_of_hammerwatch_2",
        "recent_review_date_from": None,
    },
    "necrosmith_2": {
        "appid": 2277320,
        "game_name": "Necrosmith 2",
        "game_slug": "necrosmith_2",
        "recent_review_date_from": None,
    },
    "children_of_the_sun": {
        "appid": 1309950,
        "game_name": "Children of the Sun",
        "game_slug": "children_of_the_sun",
        "recent_review_date_from": None,
    },
    "laundry_store_simulator": {
        "appid": 3150440,
        "game_name": "Laundry Store Simulator",
        "game_slug": "laundry_store_simulator",
        "recent_review_date_from": None,
    },
    "endoparasitic_2": {
        "appid": 2990640,
        "game_name": "Endoparasitic 2",
        "game_slug": "endoparasitic_2",
        "recent_review_date_from": None,
    },
}

# 04번에서 한 번에 실행할 게임 목록
TARGET_GAME_KEYS = [
    "heroes_of_hammerwatch_2",
    "necrosmith_2",
    "children_of_the_sun",
    "laundry_store_simulator",
    "endoparasitic_2",
]

# 단일 실행 모드에서 사용할 게임
TARGET_GAME_KEY = TARGET_GAME_KEYS[0]

# ============================================================
# 전처리/출시 후 산출물 기본 폴더
# ============================================================
PREPROCESS_OUTPUT_DIR = ROOT / "data" / "outputs" / "preprocess_llm"
PREPROCESSED_REVIEWS_PATH = PREPROCESS_OUTPUT_DIR / "llm_preprocessed_reviews.csv"

POSTLAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "postlaunch"
RUNS_DIR = POSTLAUNCH_OUTPUT_DIR / "runs"


def set_target_game_context(game_key):
    """게임 key를 기준으로 04번 실행에 필요한 전역 경로와 설정값을 갱신한다."""
    global TARGET_GAME_KEY, TARGET_GAME, TARGET_APPID, TARGET_GAME_NAME, GAME_SLUG, RECENT_REVIEW_DATE_FROM
    global RUN_DIR, OUTPUT_DIR, POSTLAUNCH_PREPROCESS_DIR, POSTLAUNCH_STRATEGY_DIR
    global LLM_INPUT_PATH, FILTER_LOG_PATH, SAMPLE_SUMMARY_PATH
    global CHECKPOINT_PATH, RESULT_JSON_PATH, RESULT_CSV_PATH, ISSUE_TAG_FLAT_PATH

    if game_key not in POSTLAUNCH_TARGET_GAMES:
        raise KeyError(f"POSTLAUNCH_TARGET_GAMES에 없는 game_key입니다: {game_key}")

    TARGET_GAME_KEY = game_key
    TARGET_GAME = POSTLAUNCH_TARGET_GAMES[TARGET_GAME_KEY]

    TARGET_APPID = TARGET_GAME["appid"]
    TARGET_GAME_NAME = TARGET_GAME["game_name"]
    GAME_SLUG = TARGET_GAME["game_slug"]
    RECENT_REVIEW_DATE_FROM = TARGET_GAME.get("recent_review_date_from")

    RUN_DIR = RUNS_DIR / GAME_SLUG
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # 기존 코드 호환용
    OUTPUT_DIR = RUN_DIR

    POSTLAUNCH_PREPROCESS_DIR = RUN_DIR / "postlaunch_preprocess_data"
    POSTLAUNCH_STRATEGY_DIR = RUN_DIR / "postlaunch_patch_ops_strategy_data"
    POSTLAUNCH_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)
    POSTLAUNCH_STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

    LLM_INPUT_PATH = RUN_DIR / "llm_input_reviews.csv"
    FILTER_LOG_PATH = RUN_DIR / "llm_input_filter_log.csv"
    SAMPLE_SUMMARY_PATH = RUN_DIR / "llm_input_sample_summary.csv"

    CHECKPOINT_PATH = RUN_DIR / "llm_review_analysis_checkpoint.json"
    RESULT_JSON_PATH = RUN_DIR / "llm_review_analysis_result.json"
    RESULT_CSV_PATH = RUN_DIR / "llm_review_analysis_result.csv"
    ISSUE_TAG_FLAT_PATH = RUN_DIR / "llm_issue_tags_flat.csv"

    return {
        "game_key": TARGET_GAME_KEY,
        "appid": TARGET_APPID,
        "game_name": TARGET_GAME_NAME,
        "game_slug": GAME_SLUG,
        "run_dir": RUN_DIR,
        "result_csv_path": RESULT_CSV_PATH,
        "issue_tag_flat_path": ISSUE_TAG_FLAT_PATH,
    }


# 첫 번째 게임 기준으로 경로 미리 설정
set_target_game_context(TARGET_GAME_KEY)

print("ROOT:", ROOT)
print("RUN_MULTI_GAME_MODE:", RUN_MULTI_GAME_MODE)
print("실행 대상 게임 key 목록:", TARGET_GAME_KEYS)
print("현재 미리보기 대상:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("GAME_SLUG:", GAME_SLUG)
print("최근 리뷰 시작일:", RECENT_REVIEW_DATE_FROM)
print("전처리 리뷰 파일 존재:", PREPROCESSED_REVIEWS_PATH.exists())
print("게임별 RUN_DIR:", RUN_DIR)
print("LLM 입력 저장:", LLM_INPUT_PATH)
print("LLM 결과 저장:", RESULT_CSV_PATH)
print("이슈 태그 저장:", ISSUE_TAG_FLAT_PATH)


ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
RUN_MULTI_GAME_MODE: True
실행 대상 게임 key 목록: ['heroes_of_hammerwatch_2', 'necrosmith_2', 'children_of_the_sun', 'laundry_store_simulator', 'endoparasitic_2']
현재 미리보기 대상: Heroes of Hammerwatch II
TARGET_APPID: 619820
GAME_SLUG: heroes_of_hammerwatch_2
최근 리뷰 시작일: None
전처리 리뷰 파일 존재: True
게임별 RUN_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2
LLM 입력 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_input_reviews.csv
LLM 결과 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_review_analysis_result.csv
이슈 태그 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_issue_tags_flat.csv


## Vertex AI 설정

In [3]:
# ============================================================
# Vertex AI / PydanticAI 설정
# ============================================================
# 이 셀은 실제 LLM 호출을 위한 Google Cloud 프로젝트, location, 모델명을 설정한다.

# .env 파일에 저장된 Vertex AI 설정을 현재 Python 환경으로 불러온다.
load_dotenv()
load_dotenv(ROOT / ".env", override=True)

# Google Cloud 프로젝트 ID를 읽는다.
GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

# Vertex AI location을 읽는다.
GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

# 사용할 Gemini 모델명
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

# 프로젝트 ID가 있으면 Vertex AI Provider와 Gemini 모델 객체를 생성한다.
if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    # RUN_LLM=False로 기존 결과만 읽어 후처리할 때는 프로젝트 ID가 없어도 코드을 계속 볼 수 있게 한다.
    # 단, 실제 LLM 실행 전에는 반드시 .env 또는 환경변수에 GOOGLE_CLOUD_PROJECT를 설정해야 한다.
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")


Vertex AI project: gen-lang-client-0587784564
Vertex AI location: global
Gemini model: gemini-3.1-flash-lite
Vertex 모델 생성: O


# 1. 분석/샘플링/LLM 실행 설정

In [4]:
# ============================================================
# 실행 여부
# ============================================================
RUN_LLM = False
RESET_CHECKPOINT = False
RUN_CHECK_CELLS = True

# ============================================================
# 분석 대상/샘플링/LLM 설정
# ============================================================
# TARGET_APPID, TARGET_GAME_NAME, RECENT_REVIEW_DATE_FROM은
# set_target_game_context(game_key)에서 게임별로 갱신된다.


def build_analysis_filters():
    """현재 TARGET_GAME_KEY 기준으로 분석 필터를 생성한다."""
    return {
        "appids": [TARGET_APPID],
        "game_name_contains": [],

        # 특정 게임을 appid로 직접 지정하므로 장르/카테고리/태그 제한은 두지 않는다.
        "genres": [],
        "categories": [],
        "tags": [],

        # 2024년 이후 출시 게임 중심
        "release_date_from": "2024-01-01",
        "release_date_to": None,

        # 리뷰 조건
        "languages": ["english"],
        "steam_labels": ["positive", "negative"],

        # 게임별 설정값 사용
        # None이면 보유 데이터 전체 리뷰 기준
        "review_date_from": RECENT_REVIEW_DATE_FROM,
        "review_date_to": None,

        # 출시 후 운영 판단이 목적이므로 D0-D30으로 제한하지 않는다.
        "release_periods": [],

        # 신뢰도 조건
        "steam_purchase_only": True,
        "exclude_received_for_free": True,
        "exclude_early_access_reviews": True,

        # 너무 짧거나 의미가 약한 리뷰 제외
        "meaningful_review_only": True,
    }

ANALYSIS_FILTERS = build_analysis_filters()

# ============================================================
# 샘플링 설정
# ============================================================
TEST_N = None
RANDOM_STATE = 42

# 특정 게임 1개를 대상으로 하므로 게임별 최대 리뷰 수를 최종 분석 상한처럼 사용한다.
REVIEWS_PER_GAME = 1000

# 전체 LLM 분석 리뷰 상한
MAX_TOTAL_REVIEWS = 1000

# 출시 후 패치/운영 제안은 불만 원인 파악이 중요하므로 부정 리뷰를 더 많이 포함한다.
SAMPLE_MODE = "negative_heavy_by_steam_label"
NEGATIVE_SAMPLE_RATIO = 0.7

# ============================================================
# LLM 입력 텍스트 설정
# ============================================================
MIN_REVIEW_LEN = 20
MAX_REVIEW_CHARS = 1200

# ============================================================
# LLM 호출 설정
# ============================================================
BATCH_SIZE = 3
MAX_CONCURRENT = 1
MAX_RETRIES = 3
REQUEST_SLEEP_SEC = 1
CHUNK_SIZE = MAX_CONCURRENT * 3

# ============================================================
# 저장 옵션
# ============================================================
SAVE_RESULT_JSON = True

# ============================================================
# 비용 추정 옵션
# ============================================================
INPUT_PRICE_PER_1M = 0.30
OUTPUT_PRICE_PER_1M = 2.50
USD_TO_KRW = 1500

print("RUN_LLM:", RUN_LLM)
print("RUN_MULTI_GAME_MODE:", RUN_MULTI_GAME_MODE)
print("분석 대상 게임 미리보기:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("최근 리뷰 시작일:", RECENT_REVIEW_DATE_FROM)
print("SAMPLE_MODE:", SAMPLE_MODE)
print("NEGATIVE_SAMPLE_RATIO:", NEGATIVE_SAMPLE_RATIO)
print("REVIEWS_PER_GAME:", REVIEWS_PER_GAME)
print("MAX_TOTAL_REVIEWS:", MAX_TOTAL_REVIEWS)
print("저장 폴더:", RUN_DIR)


RUN_LLM: False
RUN_MULTI_GAME_MODE: True
분석 대상 게임 미리보기: Heroes of Hammerwatch II
TARGET_APPID: 619820
최근 리뷰 시작일: None
SAMPLE_MODE: negative_heavy_by_steam_label
NEGATIVE_SAMPLE_RATIO: 0.7
REVIEWS_PER_GAME: 1000
MAX_TOTAL_REVIEWS: 1000
저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2


# 2. 공통 함수

In [5]:
# pandas/numpy/Pydantic 객체를 JSON/CSV 저장 가능한 기본 타입으로 변환한다.
def to_serializable(obj):
    """JSON 저장이 어려운 pandas/numpy/Pydantic 타입을 기본 Python 타입으로 변환한다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

def safe_json_dumps(obj):
    """리스트/딕셔너리 컬럼을 CSV에 안전하게 저장하기 위한 JSON 문자열 변환 함수."""
    safe_obj = to_serializable(obj)
    return json.dumps(safe_obj, ensure_ascii=False)


def parse_issue_tags(value):
    """
    llm_issue_tags 값을 안정적으로 리스트[dict] 형태로 변환한다.

    가능한 입력 형태:
    - 이미 list인 경우
    - JSON 문자열: [{"category": "..."}]
    - Python repr 문자열: [{'category': '...'}]
    - 비어 있는 값 / NaN
    """
    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, list):
        tags = value
    elif isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in ["nan", "none", "null"]:
            return []

        try:
            tags = json.loads(text)
        except Exception:
            try:
                tags = ast.literal_eval(text)
            except Exception:
                return []
    else:
        return []

    if not isinstance(tags, list):
        return []

    normalized = []
    for tag in tags:
        if isinstance(tag, BaseModel):
            tag = tag.model_dump()
        if isinstance(tag, dict):
            normalized.append({
                "category": tag.get("category"),
                "sentiment": tag.get("sentiment"),
                "evidence": tag.get("evidence"),
            })

    return normalized


def ensure_columns(df, columns):
    """DataFrame에 필요한 컬럼이 없으면 빈 컬럼을 추가하고, 지정 순서대로 정렬한다."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[columns].copy()


def save_csv_safely(df, path, columns=None, json_cols=None, encoding="utf-8-sig"):
    """
    CSV 저장 공통 함수.
    - 결과가 비어 있어도 헤더가 있는 CSV를 저장한다.
    - 리스트/딕셔너리 컬럼은 JSON 문자열로 변환한다.
    - 저장 폴더가 없으면 생성한다.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if df is None:
        df = pd.DataFrame()

    out = df.copy()

    if columns is not None:
        out = ensure_columns(out, columns)

    for col in (json_cols or []):
        if col in out.columns:
            out[col] = out[col].apply(lambda x: safe_json_dumps(parse_issue_tags(x)))

    out.to_csv(path, index=False, encoding=encoding)
    return out

# checkpoint 저장/불러오기 함수
def load_checkpoint(path=None):
    """이전 LLM 분석 checkpoint를 불러온다."""
    if path is None:
        path = CHECKPOINT_PATH
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=None):
    """현재까지의 LLM 분석 결과를 checkpoint JSON으로 저장한다."""
    if path is None:
        path = CHECKPOINT_PATH
    safe_results = to_serializable(results)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def load_existing_results():
    """RUN_LLM=False일 때 기존 결과를 읽는다."""
    if RESULT_JSON_PATH.exists():
        with open(RESULT_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return load_checkpoint(CHECKPOINT_PATH)


# 비용/토큰 사용량 확인 함수
def print_cost_report(input_tokens, output_tokens, requests, checkpoint_count, to_process_count):
    """토큰 사용량과 예상 비용을 출력한다."""
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    total_cost = input_cost + output_cost

    print("=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"요청 수: {requests:,}")
    print(f"checkpoint에서 불러온 리뷰 수: {checkpoint_count:,}")
    print(f"이번 실행에서 새로 처리할 리뷰 수: {to_process_count:,}")
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost:,.6f}")
    print(f"예상 비용(KRW): ₩{total_cost * USD_TO_KRW:,.0f}")
    print("=" * 60)

# PydanticAI 사용량 추출 함수
def extract_usage_tokens(result):
    """
    PydanticAI 결과 객체에서 토큰 사용량을 안전하게 추출한다.

    PydanticAI/모델 버전에 따라 usage 속성명이 조금 다를 수 있어
    여러 후보 이름을 순서대로 확인한다.
    """
    input_tokens = 0
    output_tokens = 0

    try:
        usage = result.usage()

        input_tokens = (
            getattr(usage, "input_tokens", None)
            or getattr(usage, "request_tokens", None)
            or getattr(usage, "prompt_tokens", None)
            or 0
        )

        output_tokens = (
            getattr(usage, "output_tokens", None)
            or getattr(usage, "response_tokens", None)
            or getattr(usage, "completion_tokens", None)
            or 0
        )

    except Exception:
        pass

    return int(input_tokens or 0), int(output_tokens or 0)

# 필터링/문자열 검색 헬퍼 함수
def print_filter_step(log_rows, step, before, after):
    """필터링 단계별 행 수 변화를 기록한다."""
    removed = before - after
    removed_rate = removed / before if before else 0
    log_rows.append({
        "step": step,
        "before_rows": before,
        "after_rows": after,
        "removed_rows": removed,
        "removed_rate": removed_rate,
    })
    print(f"{step}: {before:,} -> {after:,} / 제거 {removed:,} ({removed_rate:.2%})")


def contains_any_text(text, keywords):
    """문자열에 키워드 중 하나라도 포함되어 있는지 확인한다."""
    if not keywords:
        return True
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(str(keyword).lower() in text for keyword in keywords)


def pydantic_list_to_dicts(items):
    """Pydantic 객체 리스트를 CSV/JSON 저장 가능한 dict 리스트로 변환한다."""
    if items is None:
        return []
    if not isinstance(items, list):
        return []

    converted = []
    for item in items:
        if isinstance(item, BaseModel):
            converted.append(item.model_dump())
        elif isinstance(item, dict):
            converted.append(item)
    return converted


# 3. 전처리 후보 데이터 로드

In [6]:
# 전처리 후보 데이터 로드
df_candidates = pd.read_csv(PREPROCESSED_REVIEWS_PATH)

for col in ["review_datetime", "release_date"]:
    if col in df_candidates.columns:
        df_candidates[col] = pd.to_datetime(df_candidates[col], errors="coerce")

if "recommendationid" in df_candidates.columns:
    df_candidates["recommendationid"] = df_candidates["recommendationid"].astype(str)

print("전처리 후보 리뷰 수:", len(df_candidates))
print("전처리 후보 게임 수:", df_candidates["appid"].nunique())
display(df_candidates.head())


# ============================================================
# boolean 컬럼 타입 안정화
# ============================================================
# CSV를 다시 읽으면 True/False 값이 문자열로 들어오는 경우가 있다.
# 이후 필터링에서 == True, != True 조건을 안정적으로 사용하기 위해
# 주요 boolean 컬럼을 True / False / NaN 형태로 정리한다.

def normalize_bool_series(s):
    """문자열/숫자/bool 형태의 값을 True/False/NaN으로 정리한다."""
    text = s.astype(str).str.lower().str.strip()

    return pd.Series(
        np.select(
            [
                text.isin(["true", "1", "yes", "y"]),
                text.isin(["false", "0", "no", "n"]),
            ],
            [True, False],
            default=np.nan,
        ),
        index=s.index,
    )

bool_cols = [
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "is_meaningful_review",
]

for bool_col in bool_cols:
    if bool_col in df_candidates.columns:
        df_candidates[bool_col] = normalize_bool_series(df_candidates[bool_col])

print("boolean 컬럼 정리 완료")
print(df_candidates[[col for col in bool_cols if col in df_candidates.columns]].dtypes)


C:\Users\joon5\AppData\Local\Temp\ipykernel_31788\3440403886.py:2: DtypeWarning: Columns (0: top_steam_tags_text, 1: hist_first_date, 2: hist_last_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df_candidates = pd.read_csv(PREPROCESSED_REVIEWS_PATH)


전처리 후보 리뷰 수: 160644
전처리 후보 게임 수: 192


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_forever_hours,playtime_stage,votes_up,weighted_vote_score,steam_purchase,received_for_free,written_during_early_access,genres_text,categories_text,top_steam_tags_text,price,price_group,review_text_clean,review_len,is_meaningful_review,meaningless_reason,hist_total_reviews,hist_positive_reviews,hist_negative_reviews,hist_first_date,hist_last_date
0,18698790,324470,SinaRun,french,2015-10-26 18:10:33,2025-11-03,-3661,pre_release,pre_release,positive,True,1.250000,3.483333,early,2,0.523810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,good game for this price,24,False,too_short,NaN,NaN,NaN,NaN,NaN
1,18699465,324470,SinaRun,english,2015-10-26 18:53:36,2025-11-03,-3661,pre_release,pre_release,positive,True,0.216667,0.216667,very_early,1,0.421372,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",119,True,meaningful,NaN,NaN,NaN,NaN,NaN
2,18699648,324470,SinaRun,english,2015-10-26 19:03:37,2025-11-03,-3661,pre_release,pre_release,positive,True,12.666667,13.616667,late,5,0.500076,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",387,True,meaningful,NaN,NaN,NaN,NaN,NaN
3,18700348,324470,SinaRun,english,2015-10-26 19:52:39,2025-11-03,-3661,pre_release,pre_release,positive,True,0.916667,7.483333,early,16,0.637511,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...,1100,True,meaningful,NaN,NaN,NaN,NaN,NaN
4,18701774,324470,SinaRun,english,2015-10-26 21:32:24,2025-11-03,-3661,pre_release,pre_release,positive,True,6.416667,8.666667,mid,4,0.495810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,It's Lit,8,False,too_short,NaN,NaN,NaN,NaN,NaN


boolean 컬럼 정리 완료
voted_up                       float64
steam_purchase                 float64
received_for_free              float64
written_during_early_access    float64
is_meaningful_review           float64
dtype: object


# 4. 분석 조건 필터링

In [7]:
# ANALYSIS_FILTERS에 입력한 조건을 실제 후보 데이터에 적용한다.
def apply_analysis_filters(df, filters):
    """LLM 실행 파일에서 분석 목적에 맞는 필터를 적용한다."""
    filtered = df.copy()
    log_rows = []

    # 0. 최소 리뷰 길이 필터
    if "review_len" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["review_len"] >= MIN_REVIEW_LEN]
        print_filter_step(log_rows, "최소 리뷰 길이 필터", before, len(filtered))

    # 0-1. 의미 있는 리뷰 필터
    if filters.get("meaningful_review_only") is True and "is_meaningful_review" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["is_meaningful_review"] == True]
        print_filter_step(log_rows, "의미 있는 리뷰 필터", before, len(filtered))

    # 1. 게임 직접 지정 필터
    if filters.get("appids"):
        before = len(filtered)
        appids = [int(x) for x in filters["appids"]]
        filtered = filtered[filtered["appid"].isin(appids)]
        print_filter_step(log_rows, "appid 필터", before, len(filtered))

    if filters.get("game_name_contains"):
        before = len(filtered)
        keywords = [str(x).lower() for x in filters["game_name_contains"]]
        filtered = filtered[filtered["game_name"].fillna("").str.lower().apply(lambda x: any(k in x for k in keywords))]
        print_filter_step(log_rows, "게임명 키워드 필터", before, len(filtered))

    # 2. 게임 속성 필터
    # 장르/카테고리/태그 텍스트에 지정한 키워드가 포함되는지 확인한다.
    if filters.get("genres"):
        before = len(filtered)
        filtered = filtered[filtered["genres_text"].apply(lambda x: contains_any_text(x, filters["genres"]))]
        print_filter_step(log_rows, "장르 필터", before, len(filtered))

    if filters.get("categories"):
        before = len(filtered)
        filtered = filtered[filtered["categories_text"].apply(lambda x: contains_any_text(x, filters["categories"]))]
        print_filter_step(log_rows, "카테고리 필터", before, len(filtered))

    if filters.get("tags"):
        before = len(filtered)
        filtered = filtered[filtered["top_steam_tags_text"].apply(lambda x: contains_any_text(x, filters["tags"]))]
        print_filter_step(log_rows, "태그 필터", before, len(filtered))

    if filters.get("release_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["release_date_from"])
        filtered = filtered[filtered["release_date"] >= start]
        print_filter_step(log_rows, "출시일 시작 필터", before, len(filtered))

    if filters.get("release_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["release_date_to"])
        filtered = filtered[filtered["release_date"] <= end]
        print_filter_step(log_rows, "출시일 종료 필터", before, len(filtered))

    # 3. 리뷰 조건 필터
    # 언어, Steam 라벨, 리뷰 작성일, 출시 기준 구간을 적용한다.
    if filters.get("languages") and "language" in filtered.columns:
        before = len(filtered)
        allowed = [x.lower() for x in filters["languages"]]
        filtered = filtered[filtered["language"].fillna("").str.lower().isin(allowed)]
        print_filter_step(log_rows, "언어 필터", before, len(filtered))

    if filters.get("steam_labels"):
        before = len(filtered)
        filtered = filtered[filtered["steam_label_text"].isin(filters["steam_labels"])]
        print_filter_step(log_rows, "Steam 라벨 필터", before, len(filtered))

    if filters.get("review_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["review_date_from"])
        filtered = filtered[filtered["review_datetime"] >= start]
        print_filter_step(log_rows, "리뷰 작성일 시작 필터", before, len(filtered))

    if filters.get("review_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["review_date_to"])
        filtered = filtered[filtered["review_datetime"] <= end]
        print_filter_step(log_rows, "리뷰 작성일 종료 필터", before, len(filtered))

    if filters.get("release_periods"):
        before = len(filtered)
        filtered = filtered[filtered["release_period"].isin(filters["release_periods"])]
        print_filter_step(log_rows, "출시 기준 리뷰 구간 필터", before, len(filtered))

    # 4. 리뷰 신뢰도 필터
    # 실제 구매 리뷰 위주로 보고, 무료 수령/얼리액세스 리뷰를 제외할 수 있다.
    if filters.get("steam_purchase_only") is True and "steam_purchase" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["steam_purchase"] == True]
        print_filter_step(log_rows, "Steam 구매 리뷰 필터", before, len(filtered))

    if filters.get("exclude_received_for_free") is True and "received_for_free" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["received_for_free"] != True]
        print_filter_step(log_rows, "무료 수령 리뷰 제외", before, len(filtered))

    if filters.get("exclude_early_access_reviews") is True and "written_during_early_access" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["written_during_early_access"] != True]
        print_filter_step(log_rows, "얼리액세스 리뷰 제외", before, len(filtered))

    return filtered.reset_index(drop=True), pd.DataFrame(log_rows)



def prepare_filtered_reviews(show_checks=True):
    """현재 게임 기준 ANALYSIS_FILTERS를 적용해 LLM 후보 리뷰를 만든다."""
    global df_filtered, filter_log

    df_filtered, filter_log = apply_analysis_filters(df_candidates, ANALYSIS_FILTERS)

    print("최종 필터링 리뷰 수:", len(df_filtered))
    print("최종 필터링 게임 수:", df_filtered["appid"].nunique())

    if show_checks:
        display(filter_log)
        print("필터링 후 release_period 분포")
        display(df_filtered["release_period"].value_counts(dropna=False))
        print("필터링 후 Steam 라벨 분포")
        display(df_filtered["steam_label_text"].value_counts(dropna=False))

    return df_filtered, filter_log


if not RUN_MULTI_GAME_MODE:
    df_filtered, filter_log = prepare_filtered_reviews(show_checks=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: 필터링은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: 필터링은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 5. 게임별 샘플링 및 LLM 입력 파일 저장

In [8]:
# 게임 1개에 대해 리뷰를 샘플링한다.
# - recent: 최신 리뷰 우선
# - random: 무작위
# - balanced_by_steam_label: 긍정/부정 리뷰를 가능하면 균형 있게 섞음
# - negative_heavy_by_steam_label: 부정 리뷰를 더 많이 포함하도록 섞음
def sample_one_game(group, reviews_per_game, mode, random_state):
    """
    게임 1개에 대해 리뷰를 샘플링하는 함수.
    샘플링 방식은 분석 목적에 따라 LLM 실행 파일에서 선택한다.
    """
    if reviews_per_game is None or len(group) <= reviews_per_game:
        return group

    if mode == "recent":
        return group.sort_values("review_datetime", ascending=False).head(reviews_per_game)

    if mode == "random":
        return group.sample(n=reviews_per_game, random_state=random_state)

    if mode == "balanced_by_steam_label":
        half = reviews_per_game // 2

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        pos_n = min(len(positive), half)
        neg_n = min(len(negative), reviews_per_game - pos_n)

        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)
        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)

        sampled = pd.concat([pos_sample, neg_sample], axis=0)

        # 한쪽 라벨이 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    if mode == "negative_heavy_by_steam_label":
        target_neg_n = int(np.ceil(reviews_per_game * NEGATIVE_SAMPLE_RATIO))

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        neg_n = min(len(negative), target_neg_n)
        pos_n = min(len(positive), reviews_per_game - neg_n)

        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)
        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)

        sampled = pd.concat([neg_sample, pos_sample], axis=0)

        # 부정/긍정 리뷰가 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    raise ValueError(f"지원하지 않는 SAMPLE_MODE입니다: {mode}")

# 게임별 샘플링 실행
def sample_reviews_per_game(df, reviews_per_game, mode, random_state):
    sampled_groups = []

    for _, group in df.groupby("appid", group_keys=False):
        sampled_groups.append(sample_one_game(group, reviews_per_game, mode, random_state))

    if not sampled_groups:
        return df.head(0)

    sampled = pd.concat(sampled_groups, axis=0).reset_index(drop=True)

    if MAX_TOTAL_REVIEWS is not None and len(sampled) > MAX_TOTAL_REVIEWS:
        sampled = sampled.sample(n=MAX_TOTAL_REVIEWS, random_state=random_state).reset_index(drop=True)

    if TEST_N is not None:
        sampled = sampled.head(TEST_N).copy()

    return sampled




def prepare_llm_input_for_current_game(show_preview=True):
    """현재 게임의 필터링 결과를 샘플링하고 LLM 입력 파일을 저장한다."""
    global df_sampled, df_for_llm, sample_summary
    df_sampled = sample_reviews_per_game(
        df_filtered,
        reviews_per_game=REVIEWS_PER_GAME,
        mode=SAMPLE_MODE,
        random_state=RANDOM_STATE,
    )

    # LLM에 전달할 텍스트 길이 제한은 LLM 실행 파일에서 적용한다.
    df_sampled["review_text_for_llm"] = df_sampled["review_text_clean"].fillna("").astype(str).str.slice(0, MAX_REVIEW_CHARS)

    # LLM 입력 파일 컬럼 구성
    # 프롬프트에 필요한 리뷰/게임 정보와 후속 결과 매칭에 필요한 키만 남긴다.
    LLM_INPUT_COLUMNS = [
        # 원본 리뷰 식별 정보
        "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
        "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
        "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 전달합니다.
        "language",                      # 리뷰 작성 언어입니다. 현재 분석에서는 영어 리뷰 필터링 여부 확인에 사용합니다.

        # 리뷰 작성 시점 / 출시 후 구간 정보
        "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 어느 시점의 반응인지 확인할 때 사용합니다.
        "release_date",                  # 게임 출시일입니다. 리뷰 작성일과 비교해 출시 후 경과일을 계산할 때 사용합니다.
        "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
        "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.
        "release_period_detail",         # 출시 후 구간을 더 세부적으로 나눈 값입니다. 초기 반응을 더 자세히 볼 때 사용합니다.

        # Steam 원본 라벨 / 리뷰 메타 정보
        "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
        "voted_up",                      # Steam 원본 추천 여부입니다. True면 추천, False면 비추천 리뷰입니다.
        "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
        "playtime_stage",                # 플레이타임을 구간화한 값입니다. 초반/중반/장기 플레이 리뷰를 구분할 때 사용합니다.
        "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
        "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.
        "received_for_free",             # 무료로 받은 게임인지 여부입니다. 일반 구매자 반응과 구분할 때 사용합니다.
        "written_during_early_access",   # 얼리액세스 기간에 작성된 리뷰인지 여부입니다.

        # 게임 메타 정보
        "price",                         # 게임 가격입니다. 가격대별 반응 분석에 사용합니다.
        "price_group",                   # 가격을 구간으로 나눈 값입니다. 가격대별 비교에 사용합니다.
        "genres_text",                   # 게임 장르 목록입니다. 장르별 반응 분석에 사용합니다.
        "categories_text",               # 게임 카테고리 목록입니다. 싱글/멀티/협동 여부 분석에 사용합니다.
        "top_steam_tags_text",           # 주요 Steam 태그 목록입니다. 태그별 반응 비교에 사용합니다.

        # LLM 입력 텍스트 / 유효성 판단 정보
        "review_text_for_llm",           # LLM에 실제로 전달할 리뷰 본문입니다. 전처리된 텍스트를 사용합니다.
        "is_meaningful_review",          # 분석에 의미 있는 리뷰인지 여부입니다. 무의미한 리뷰를 제외하거나 별도 확인할 때 사용합니다.
        "meaningless_reason",            # 무의미한 리뷰로 판단된 이유입니다. 예: 짧은 문장, 이모지만 존재, 내용 없음 등입니다.
    ]

    input_cols = [c for c in LLM_INPUT_COLUMNS if c in df_sampled.columns]
    df_for_llm = df_sampled[input_cols].copy()

    # 최종 LLM 입력 파일과 필터 로그를 저장한다.
    # 이 파일을 보면 어떤 리뷰가 실제 분석 대상이 되었는지 재현할 수 있다.
    df_for_llm.to_csv(LLM_INPUT_PATH, index=False, encoding="utf-8-sig")
    filter_log.to_csv(FILTER_LOG_PATH, index=False, encoding="utf-8-sig")

    # 게임별로 최종 샘플 리뷰 수와 라벨 분포를 요약한다.
    sample_summary = (
        df_for_llm
        .groupby(["appid", "game_name"], as_index=False)
        .agg(
            sampled_review_count=("recommendationid", "count"),
            positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
            negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
            first_review_datetime=("review_datetime", "min"),
            last_review_datetime=("review_datetime", "max"),
        )
        .sort_values("sampled_review_count", ascending=False)
    )
    sample_summary.to_csv(SAMPLE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    print("LLM 입력 리뷰 수:", len(df_for_llm))
    print("LLM 입력 게임 수:", df_for_llm["appid"].nunique())
    print("LLM 입력 저장:", LLM_INPUT_PATH)
    print("필터 로그 저장:", FILTER_LOG_PATH)
    print("샘플 요약 저장:", SAMPLE_SUMMARY_PATH)

    if show_preview:
        display(df_for_llm.head())
        display(sample_summary.head())

    return df_for_llm, sample_summary


if not RUN_MULTI_GAME_MODE:
    df_for_llm, sample_summary = prepare_llm_input_for_current_game(show_preview=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: LLM 입력 파일 생성은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: LLM 입력 파일 생성은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 6. PydanticAI 출력 스키마 정의

- 결과 컬럼이 매번 달라지는 것을 방지한다.
- JSON 파싱 오류를 줄인다.
- 후속 집계표를 안정적으로 만들 수 있다.
- `category` 목록은 이후 이슈 집계와 시각화의 기준이 된다.

## `llm_urgency_candidate` 해석 기준

이 단계의 `llm_urgency_candidate`는 LLM이 리뷰 문맥을 보고 판단한 **시급도 후보**다.  
예를 들어 크래시, 저장 오류, 진행 불가처럼 출시 후 플레이 경험에 큰 영향을 줄 수 있는 내용은 높게 분류될 수 있다.

다만 이 값은 **최종 개선 우선순위가 아니다.**  
최종 패치·운영 우선순위는 이후 집계 단계에서 이슈 반복성, 부정 맥락, 최근성, 플레이타임 구간 등을 함께 고려해 데이터 기준으로 산정한다.


| 설정값 | 의미 |
|---|---|
| `bug` | 버그 |
| `optimization` | 최적화 |
| `performance` | 성능/프레임 |
| `crash` | 튕김/실행 불가 |
| `control` | 조작감 |
| `balance` | 밸런스 |
| `difficulty` | 난이도 |
| `content_volume` | 콘텐츠 양 |
| `story` | 스토리 |
| `translation_localization` | 번역/현지화 |
| `ui_ux` | UI/UX |
| `price_value` | 가격 대비 가치 |
| `multiplayer_network` | 멀티/서버 |
| `save_progression` | 저장/진행도 |
| `graphics_audio` | 그래픽/사운드 |
| `gameplay_loop` | 핵심 재미/반복 구조 |
| `monetization` | 과금/DLC |
| `developer_communication` | 개발자 소통 |
| `positive_praise` | 전반적 칭찬 |
| `progression_grind` | 성장/노가다 |
| `other` | 기타 |

In [9]:
# ============================================================
# LLM 출력 스키마
# ============================================================
# PydanticAI의 output_type으로 사용할 Pydantic 모델이다.
# 이 스키마를 기준으로 LLM 출력이 구조화되어 들어온다.
# ============================================================

class IssueTag(BaseModel):
    category: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰에서 언급된 세부 이슈 카테고리")

    # sentiment는 해당 이슈에 대한 감정 방향이다.
    # 리뷰 전체 감정이 아니라, 이 세부 이슈 하나에 대한 감정이다.
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    # evidence는 왜 이 카테고리/감정으로 판단했는지에 대한 짧은 근거다.
    # 원문 리뷰를 바탕으로 작성된다.
    evidence: str = Field(
        description="원문 리뷰를 바탕으로 한 짧은 판단 근거",
        min_length=1,
        max_length=160,
    )


class SteamReviewAnalysis(BaseModel):
    # 입력 리뷰 ID를 그대로 반환한다.
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    # LLM이 리뷰 본문만 보고 판단한 감정이다.
    # Steam의 voted_up과 다를 수 있다.
    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    # 감정을 1~5점으로 수치화한 값이다.
    # 1은 매우 부정, 3은 중립, 5는 매우 긍정으로 해석한다.
    sentiment_score: int = Field(
        ge=1,
        le=5,
        description="1=매우 부정, 3=중립, 5=매우 긍정",
    )

    llm_primary_issue: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰의 대표 이슈")

    # 리뷰 안에서 발견된 여러 세부 이슈 목록이다.
    llm_issue_tags: List[IssueTag] = Field(
        default_factory=list,
        description="리뷰에서 발견된 세부 이슈 목록",
    )

    # 리뷰 문맥상 문제 강도 후보이다. 최종 패치·운영 우선순위는 아니다.
    llm_urgency_candidate: Literal["low", "medium", "high"] = Field(
        description="LLM이 리뷰 문맥을 보고 분류한 시급도 후보. 최종 우선순위는 아님"
    )

    # 리뷰 핵심 내용 요약이다.
    llm_review_summary: str = Field(
        description="리뷰 핵심 내용 요약",
        min_length=5,
        max_length=220,
    )

    # 리뷰 내용을 바탕으로 정리한 개선 방향 후보이다.
    llm_suggested_action: str = Field(
        description="리뷰 내용을 바탕으로 정리한 개선 방향 후보",
        min_length=5,
        max_length=260,
    )


class BatchSteamReviewAnalysis(BaseModel):
    """한 번의 배치 요청에서 여러 리뷰 결과를 받을 수 있도록 감싸는 모델."""
    results: List[SteamReviewAnalysis] = Field(
        description="리뷰별 분석 결과 목록"
    )


ISSUE_KR_MAP = {
    "bug": "버그",
    "optimization": "최적화",
    "performance": "성능",
    "crash": "크래시",
    "control": "조작감",
    "balance": "밸런스",
    "difficulty": "난이도",
    "content_volume": "콘텐츠 분량",
    "story": "스토리",
    "translation_localization": "번역/현지화",
    "ui_ux": "UI/UX",
    "price_value": "가격/가치",
    "multiplayer_network": "멀티/네트워크",
    "save_progression": "저장/진행",
    "graphics_audio": "그래픽/사운드",
    "gameplay_loop": "게임플레이 루프",
    "monetization": "과금",
    "developer_communication": "개발사 소통",
    "positive_praise": "긍정 칭찬",
    "progression_grind": "성장/반복 노가다",
    "other": "기타",
}


# 7. 프롬프트 및 PydanticAI Agent 설정
LLM에게 어떤 역할을 부여할지, 어떤 모델 설정으로 호출할지 정한다.

| 설정값 | 의미 |
|---|---|
| `system_prompt` | LLM에게 부여하는 분석 기준과 제한 사항 |
| `temperature=0.0` | 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 설정 |
| `review_agent` | PydanticAI가 Vertex AI Gemini를 호출할 때 사용하는 Agent |



In [10]:
system_prompt = """
당신은 Steam 인디게임 출시 후 리뷰를 분석해 패치·운영 방향을 제안하는 전문가입니다.

각 리뷰에 대해 다음을 분류/작성하세요.
1. 리뷰 본문 기준 감정(llm_sentiment)
2. 가장 핵심적인 이슈(llm_primary_issue)
3. 세부 이슈(llm_issue_tags)
4. 리뷰 내용에 근거한 개발사 참고용 개선 방향 후보(llm_suggested_action)
5. 리뷰 문맥상 문제 강도 후보(llm_urgency_candidate)

주의:
- llm_urgency_candidate는 최종 개선 우선순위가 아니라, 리뷰 문맥에서 문제가 강하게 표현되었는지 확인하기 위한 보조 분류값입니다.
- 최종 패치·운영 우선순위는 후속 집계에서 이슈 반복성, 부정 맥락, 최근성, 플레이타임 구간 등을 기준으로 별도 계산합니다.

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- llm_issue_tags에는 실제로 언급된 것만 넣으세요.
- evidence는 리뷰 본문에 근거가 있는 짧은 표현 또는 요약으로 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
- 분석할 정보가 부족한 리뷰는 llm_primary_issue를 other로 두고, llm_urgency_candidate는 low로 분류하세요.
- 리뷰 본문에 근거가 없는 개선 제안은 작성하지 말고 보수적으로 작성하세요.
- 게임 메타데이터와 태그는 맥락 참고용이며, 리뷰 본문에 없는 내용을 억지로 추론하지 마세요.
- llm_primary_issue가 positive_praise인 경우, llm_urgency_candidate는 원칙적으로 low로 분류하세요.
- 긍정 리뷰의 suggested_action은 문제 해결 지시가 아니라 유지/강화할 강점 중심으로 작성하세요.
- sentiment_score는 llm_sentiment와 일관되게 작성하세요.
- negative는 1~2, neutral은 3, mixed는 2~4 범위에서 맥락에 맞게, positive는 4~5로 작성하세요.
- llm_suggested_action은 최종 전략이 아니라, 리뷰 본문에 근거한 개발사 참고용 개선 방향 후보로 작성하세요.
- 인디게임 개발사가 출시 후 패치·운영에서 실제로 참고할 수 있게 구체적으로 작성하되, 리뷰에 없는 위험을 새로 만들지 마세요.
- llm_urgency_candidate를 이용해 전체 우선순위를 정하거나, 리뷰에 없는 위험을 새로 추론하지 마세요.
"""

# Gemini 모델 세부 설정
# temperature=0.0으로 두어 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 한다.
review_settings = GoogleModelSettings(
    temperature=0.0,
)

# Vertex 모델이 정상 생성된 경우에만 PydanticAI Agent를 만든다.
# RUN_LLM=False로 기존 결과만 읽을 때는 Agent가 없어도 후처리 셀을 볼 수 있다.
if vertex_model is not None:
    review_agent = Agent(
        vertex_model,
        output_type=BatchSteamReviewAnalysis,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    review_agent = None

print("PydanticAI Agent 생성 여부:", "O" if review_agent is not None else "X")

PydanticAI Agent 생성 여부: O


# 8. 프롬프트 생성 함수


In [11]:
def build_batch_prompt(batch_df):
    """
    여러 개의 리뷰를 한 번에 LLM에게 보내기 위한 프롬프트 생성 함수.
    """
    blocks = []

    for _, row in batch_df.iterrows():
        block = f"""
[REVIEW]
recommendationid: {row["recommendationid"]}
appid: {row["appid"]}
game_name: {row.get("game_name", "")}
genres: {row.get("genres_text", "")}
categories: {row.get("categories_text", "")}
steam_top_tags: {row.get("top_steam_tags_text", "")}
release_date: {row.get("release_date", "")}
review_datetime: {row.get("review_datetime", "")}
days_from_release: {row.get("days_from_release", "")}
release_period: {row.get("release_period", "")}
language: {row.get("language", "")}
voted_up: {row.get("voted_up", "")}
steam_label_text: {row.get("steam_label_text", "")}
playtime_at_review_hours: {row.get("playtime_at_review_hours", "")}
playtime_stage: {row.get("playtime_stage", "")}
votes_up: {row.get("votes_up", "")}
weighted_vote_score: {row.get("weighted_vote_score", "")}
received_for_free: {row.get("received_for_free", "")}
written_during_early_access: {row.get("written_during_early_access", "")}

review:
{row.get("review_text_for_llm", "")}
[/REVIEW]
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\\n"
        "결과는 지정된 Pydantic 스키마에 맞게 반환하세요.\\n\\n"
        + "\\n".join(blocks)
    )

    return prompt


# 9. PydanticAI Vertex LLM 호출 함수

실제 Vertex AI Gemini 호출을 수행하는 함수

핵심 흐름은 다음과 같다.
1. 리뷰 배치를 프롬프트로 변환한다.
2. PydanticAI Agent로 Vertex AI Gemini를 호출한다.
3. LLM이 반환한 결과를 `recommendationid` 기준으로 원본 리뷰와 매칭한다.
4. 성공/누락/실패 결과를 모두 기록한다.
5. 중간 결과를 checkpoint에 저장해 실행 중단 시에도 복구할 수 있게 한다.

AI와 씨름한 결과물이라 이게 맞는건지는....

In [12]:
# 동시에 실행될 LLM 요청 수를 제한하기 위한 Semaphore
# MAX_CONCURRENT=1이면 한 번에 요청 1개만 실행
sem = asyncio.Semaphore(MAX_CONCURRENT)

# 리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석
# 1. batch_df를 LLM 프롬프트로 변환
# 2. Vertex AI Gemini 호출
# 3. Pydantic 구조로 받은 결과를 원본 리뷰와 매칭
# 4. 결과를 all_results에 추가
# 5. 실패하면 재시도하고, 최종 실패 시 실패 기록을 남김
async def analyze_batch(batch_df, all_results, stats, pbar):
    """
    리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석한다.
    """
    async with sem:
        prompt = build_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                if review_agent is None:
                    raise RuntimeError(
                        "review_agent가 생성되지 않았습니다. "
                        ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
                    )

                result = await review_agent.run(
                    prompt,
                    model_settings=review_settings,
                )

                # PydanticAI가 스키마에 맞춰 구조화한 결과를 가져온다.
                output = result.output
                output_items = getattr(output, "results", [])

                # 토큰 사용량을 누적해 예상 비용을 확인
                input_tokens, output_tokens = extract_usage_tokens(result)
                stats["input_tokens"] += input_tokens
                stats["output_tokens"] += output_tokens
                stats["requests"] += 1

                # 입력 리뷰 ID와 LLM 반환 ID를 비교해 누락/오반환 여부를 확인
                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output_items:
                    rid = str(item.recommendationid)

                    # LLM이 입력에 없던 ID를 반환하면 무시한다.
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    record = {
                        # 중요
                        # "llm_primary_issue": 부정/긍정 반응의 핵심 원인 분류입니다.
                        # "llm_urgency_candidate": LLM이 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
                        # "llm_suggested_action": 리뷰 내용을 바탕으로 정리한 개선 방향 후보입니다.
                        "analysis_status": "success",  # LLM 분석이 정상적으로 완료된 리뷰임을 표시합니다.

                        # 원본 리뷰 식별 정보
                        "recommendationid": rid,  # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
                        "appid": row.get("appid"),  # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
                        "game_name": row.get("game_name", ""),  # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.
                        "review_datetime": row.get("review_datetime", None),  # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
                        "release_date": row.get("release_date", None),  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
                        "days_from_release": row.get("days_from_release", None),  # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
                        "release_period": row.get("release_period", None),  # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

                        # Steam 라벨/리뷰 메타
                        "steam_label_text": row.get("steam_label_text", ""),  # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),  # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
                        "votes_up": row.get("votes_up", None),  # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
                        "weighted_vote_score": row.get("weighted_vote_score", None),  # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

                        # LLM 분석 결과
                        "llm_sentiment": item.llm_sentiment,  # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
                        "sentiment_score": item.sentiment_score,  # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
                        "llm_primary_issue": item.llm_primary_issue,  # LLM이 판단한 리뷰의 대표 이슈입니다. 버그/밸런스/콘텐츠/가격 등 주요 원인을 나타냅니다.
                        "llm_issue_tags": pydantic_list_to_dicts(item.llm_issue_tags),  # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
                        "llm_urgency_candidate": item.llm_urgency_candidate,  # LLM이 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
                        "llm_review_summary": item.llm_review_summary,  # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
                        "llm_suggested_action": item.llm_suggested_action,  # LLM이 제안한 개선 방향입니다. 패치/운영 방향 제안에 활용합니다.
                    }

                    all_results.append(record)

                # LLM이 누락한 리뷰가 있으면 누락 기록을 남긴다.
                missing_ids = input_ids - matched_ids
                for rid in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    all_results.append({
                        "analysis_status": "missing_in_llm_output",
                        "recommendationid": rid,
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "llm_primary_issue": None,
                        "llm_issue_tags": [],
                        "llm_urgency_candidate": None,
                        "llm_review_summary": None,
                        "llm_suggested_action": None,
                    })

                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_sec = 2 ** attempt
                    print(f"배치 분석 실패, 재시도 {attempt + 1}/{MAX_RETRIES}: {e}")
                    await asyncio.sleep(wait_sec)
                else:
                    print(f"배치 최종 실패: {e}")

                    for _, row in batch_df.iterrows():
                        all_results.append({
                            "analysis_status": "failed",
                            "recommendationid": str(row.get("recommendationid")),
                            "appid": row.get("appid"),
                            "game_name": row.get("game_name", ""),
                            "steam_label_text": row.get("steam_label_text", ""),
                            "llm_sentiment": None,
                            "sentiment_score": None,
                            "llm_primary_issue": None,
                            "llm_issue_tags": [],
                            "llm_urgency_candidate": None,
                            "llm_review_summary": None,
                            "llm_suggested_action": None,
                            "error_message": str(e),
                        })

                    save_checkpoint(all_results)
                    pbar.update(len(batch_df))
                    return

# 전체 LLM 분석을 실행한다.
# 처리 흐름:
# 1. 기존 checkpoint를 읽는다.
# 2. 현재 분석 대상 ID만 checkpoint에서 유지한다.
# 3. 이미 처리된 recommendationid는 제외한다.
# 4. 남은 리뷰를 BATCH_SIZE 단위로 나눈다.
# 5. CHUNK_SIZE 단위로 비동기 요청을 실행한다.
# 6. 중간 결과는 checkpoint에 계속 저장한다.
async def run_analysis(df):
    """
    전체 LLM 분석을 실행한다.
    """
    if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("기존 checkpoint 삭제:", CHECKPOINT_PATH)

    all_results = load_checkpoint()
    all_results = list(all_results)

    # 현재 분석 대상 ID만 checkpoint에서 유지한다.
    target_ids = set(df["recommendationid"].astype(str))
    all_results = [
        row for row in all_results
        if str(row.get("recommendationid")) in target_ids
    ]

    done_ids = {
        str(row.get("recommendationid"))
        for row in all_results
        if row.get("analysis_status") in ["success", "missing_in_llm_output", "failed"]
    }

    to_process = df[~df["recommendationid"].astype(str).isin(done_ids)].copy()

    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(done_ids),
        "to_process_count": len(to_process),
    }

    if len(to_process) == 0:
        print("새로 처리할 리뷰가 없습니다. checkpoint 또는 기존 결과를 사용합니다.")
        return all_results, stats

    batches = [
        to_process.iloc[i:i + BATCH_SIZE]
        for i in range(0, len(to_process), BATCH_SIZE)
    ]

    with tqdm(total=len(to_process), desc="PydanticAI Vertex 리뷰 분석 진행") as pbar:
        for start in range(0, len(batches), CHUNK_SIZE):
            chunk = batches[start:start + CHUNK_SIZE]

            tasks = [
                analyze_batch(batch_df, all_results, stats, pbar)
                for batch_df in chunk
            ]

            await asyncio.gather(*tasks)

            if REQUEST_SLEEP_SEC > 0:
                await asyncio.sleep(REQUEST_SLEEP_SEC)

    return all_results, stats


# 10. LLM 분석 실행

`RUN_LLM` 설정에 따라 실제 LLM 호출 여부가 달라진다.

- `RUN_LLM=True`: Vertex AI Gemini를 실제 호출한다.
- `RUN_LLM=False`: 기존 JSON/checkpoint 결과를 읽어 후처리만 수행한다.

실행 후에는 토큰 사용량과 예상 비용을 출력한다.

In [13]:
async def execute_llm_analysis_for_current_game(show_preview=True):
    """현재 게임의 df_for_llm을 기준으로 LLM 분석을 실행하거나 기존 결과를 불러온다."""
    global results, stats, safe_results, df_result_raw

    if RUN_LLM:
        results, stats = await run_analysis(df_for_llm)
    else:
        results = load_existing_results()
        stats = {
            "input_tokens": 0,
            "output_tokens": 0,
            "requests": 0,
            "checkpoint_count": len(results),
            "to_process_count": 0,
        }

    print_cost_report(
        input_tokens=stats["input_tokens"],
        output_tokens=stats["output_tokens"],
        requests=stats["requests"],
        checkpoint_count=stats["checkpoint_count"],
        to_process_count=stats["to_process_count"],
    )

    safe_results = to_serializable(results)
    df_result_raw = pd.DataFrame(safe_results)

    print("분석 결과 행 수:", len(df_result_raw))
    if show_preview:
        display(df_result_raw.head())

    return results, stats, df_result_raw


if not RUN_MULTI_GAME_MODE:
    results, stats, df_result_raw = await execute_llm_analysis_for_current_game(show_preview=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: LLM 분석 실행은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: LLM 분석 실행은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 11. 리뷰 단위 결과 후처리 및 저장


In [14]:
def classify_sentiment_relation(row):
    """Steam 라벨과 LLM 감정의 관계 분류"""
    steam_label = row.get("steam_label_text")
    llm_sentiment = row.get("llm_sentiment")

    if steam_label == "positive":
        if llm_sentiment == "positive":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "negative":
            return "mismatch"
        return "unclear"

    if steam_label == "negative":
        if llm_sentiment == "negative":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "positive":
            return "mismatch"
        return "unclear"

    return "unknown"

# 리뷰 단위 결과 저장 컬럼
REVIEW_RESULT_COLUMNS = [
    # 분석 상태
    "analysis_status",               # LLM 분석 성공/실패 여부를 표시합니다.

    # 원본 리뷰 식별 정보
    "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
    "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
    "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.

    # 리뷰 작성 시점 / 출시 후 구간 정보
    "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
    "release_date",                  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
    "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
    "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

    # Steam 라벨 / 리뷰 메타 정보
    "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
    "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
    "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
    "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

    # LLM 분석 결과
    "llm_sentiment",                 # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
    "sentiment_score",               # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
    "llm_primary_issue",                 # LLM이 판단한 리뷰의 대표 이슈입니다. 부정/긍정 반응의 핵심 원인 분류에 사용합니다.
    "llm_issue_tags",                    # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
    "llm_urgency_candidate",                       # LLM이 리뷰 문맥 기준으로 분류한 시급도 후보입니다. 최종 우선순위가 아니라 보조 참고용입니다.
    "llm_review_summary",                       # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
    "llm_suggested_action",              # LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보입니다.

    # Steam 라벨과 LLM 판단 비교 결과
    "steam_llm_sentiment_relation",            # Steam 추천/비추천 라벨과 LLM 감정 판단이 어느 정도 일치하는지 비교한 값입니다.
]



def postprocess_and_save_review_results(show_preview=False):
    """df_result_raw를 리뷰 단위 결과 파일로 저장한다."""
    global df_result_all, df_result, df_failed_log, df_result_save
    # 결과 후처리
    # df_result: 보고서에서 사용할 성공 분석 결과만 포함한 결과
    # df_failed_log: 실패/누락 결과 확인용 메모리 데이터프레임
    if len(df_result_raw) > 0:
        df_result_all = df_result_raw.copy()

        # 필수 컬럼 보강
        df_result_all = ensure_columns(
            df_result_all,
            list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"]))
        )

        # 타입 안정화
        df_result_all["recommendationid"] = df_result_all["recommendationid"].astype(str)

        for col in ["review_datetime", "release_date"]:
            if col in df_result_all.columns:
                df_result_all[col] = pd.to_datetime(df_result_all[col], errors="coerce")

        # llm_issue_tags는 메모리에서는 list[dict] 형태로 정규화
        df_result_all["llm_issue_tags"] = df_result_all["llm_issue_tags"].apply(parse_issue_tags)

        if {"steam_label_text", "llm_sentiment"}.issubset(df_result_all.columns):
            df_result_all["steam_llm_sentiment_relation"] = df_result_all.apply(classify_sentiment_relation, axis=1)
        else:
            df_result_all["steam_llm_sentiment_relation"] = "unknown"

        if "analysis_status" in df_result_all.columns:
            df_result = df_result_all[df_result_all["analysis_status"] == "success"].copy()
            df_failed_log = df_result_all[df_result_all["analysis_status"] != "success"].copy()
        else:
            df_result = df_result_all.copy()
            df_failed_log = pd.DataFrame(columns=df_result_all.columns)

    else:
        df_result_all = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))
        df_result = pd.DataFrame(columns=REVIEW_RESULT_COLUMNS)
        df_failed_log = pd.DataFrame(columns=list(dict.fromkeys(REVIEW_RESULT_COLUMNS + ["error_message"])))
        print("분석 결과가 없습니다. 빈 결과 파일을 헤더만 포함해 저장합니다.")


    # 저장용 데이터프레임 생성
    # 최종 보고서에서 사용하는 핵심 산출물만 저장한다.
    df_result_save = ensure_columns(df_result, REVIEW_RESULT_COLUMNS)

    # 성공 결과 CSV 저장
    # llm_issue_tags는 보고서 코드에서 다시 안전하게 읽을 수 있도록 JSON 문자열로 저장한다.
    df_result_save = save_csv_safely(
        df_result_save,
        RESULT_CSV_PATH,
        columns=REVIEW_RESULT_COLUMNS,
        json_cols=["llm_issue_tags"],
    )

    # 성공 결과 JSON 저장
    if SAVE_RESULT_JSON:
        safe_success_results = to_serializable(df_result.to_dict(orient="records"))
        RESULT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(safe_success_results, f, ensure_ascii=False, indent=2)
        print("리뷰 단위 LLM 결과 JSON 저장:", RESULT_JSON_PATH)

    print("리뷰 단위 LLM 결과 CSV 저장:", RESULT_CSV_PATH)
    print("성공 분석 리뷰 수:", len(df_result))
    print("실패/누락 리뷰 수:", len(df_failed_log))
    print("전체 결과 행 수:", len(df_result_all))

    return df_result, df_failed_log, df_result_all


if not RUN_MULTI_GAME_MODE:
    df_result, df_failed_log, df_result_all = postprocess_and_save_review_results(show_preview=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: 리뷰 단위 결과 저장은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: 리뷰 단위 결과 저장은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 12. 이슈 태그 펼치기


In [15]:
# 이슈 태그 펼친 결과 저장 컬럼
ISSUE_TAG_FLAT_COLUMNS = [
    "recommendationid",
    "appid",
    "game_name",
    "steam_label_text",
    "llm_sentiment",
    "llm_primary_issue",
    "llm_urgency_candidate",
    "release_period",
    "playtime_at_review_hours",
    "votes_up",
    "weighted_vote_score",
    "llm_issue_category",
    "issue_name_kor",
    "llm_issue_sentiment",
    "llm_issue_evidence",
]


def flatten_issue_tags(df):
    """
    리뷰별 llm_issue_tags 리스트를 이슈 단위 행으로 펼친다.
    """
    flat_rows = []

    if len(df) == 0 or "llm_issue_tags" not in df.columns:
        return pd.DataFrame(columns=ISSUE_TAG_FLAT_COLUMNS)

    for _, row in df.iterrows():
        tags = row.get("llm_issue_tags", [])

        if isinstance(tags, str):
            try:
                tags = json.loads(tags)
            except Exception:
                tags = []

        if not isinstance(tags, list):
            continue

        for tag in tags:
            if not isinstance(tag, dict):
                continue

            category = tag.get("category")
            issue_name_kor = ISSUE_KR_MAP.get(category, category)

            flat_rows.append({
                "recommendationid": row.get("recommendationid"),
                "appid": row.get("appid"),
                "game_name": row.get("game_name"),
                "steam_label_text": row.get("steam_label_text"),
                "llm_sentiment": row.get("llm_sentiment"),
                "llm_primary_issue": row.get("llm_primary_issue"),
                "llm_urgency_candidate": row.get("llm_urgency_candidate"),
                "release_period": row.get("release_period"),
                "playtime_at_review_hours": row.get("playtime_at_review_hours"),
                "votes_up": row.get("votes_up"),
                "weighted_vote_score": row.get("weighted_vote_score"),
                "llm_issue_category": category,
                "issue_name_kor": issue_name_kor,
                "llm_issue_sentiment": tag.get("sentiment"),
                "llm_issue_evidence": tag.get("evidence"),
            })

    return pd.DataFrame(flat_rows, columns=ISSUE_TAG_FLAT_COLUMNS)




def save_issue_tags_flat_for_current_game(show_preview=True):
    """리뷰 단위 LLM 결과의 llm_issue_tags를 이슈 태그 단위로 펼쳐 저장한다."""
    global df_issue_tags_flat
    df_issue_tags_flat = flatten_issue_tags(df_result)
    df_issue_tags_flat.to_csv(ISSUE_TAG_FLAT_PATH, index=False, encoding="utf-8-sig")

    print("이슈 태그 펼친 결과 저장:", ISSUE_TAG_FLAT_PATH)
    print("이슈 태그 행 수:", len(df_issue_tags_flat))
    if show_preview:
        display(df_issue_tags_flat.head())

    return df_issue_tags_flat


if not RUN_MULTI_GAME_MODE:
    df_issue_tags_flat = save_issue_tags_flat_for_current_game(show_preview=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: 이슈 태그 펼치기는 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: 이슈 태그 펼치기는 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 13. 산출물 점검


In [16]:
def check_output_files_for_current_game(show_preview=True):
    """현재 게임의 04번 산출물 생성 여부와 행/컬럼 수를 확인한다."""
    global output_check
    output_check_targets = {
        "LLM 입력 리뷰 CSV": LLM_INPUT_PATH,
        "필터 로그 CSV": FILTER_LOG_PATH,
        "게임별 샘플 요약 CSV": SAMPLE_SUMMARY_PATH,
        "리뷰별 LLM 분석 결과 CSV": RESULT_CSV_PATH,
        "이슈 태그 펼친 결과 CSV": ISSUE_TAG_FLAT_PATH,
        "LLM 분석 중간 저장 파일": CHECKPOINT_PATH,
    }

    if SAVE_RESULT_JSON:
        output_check_targets["리뷰별 LLM 분석 결과 JSON"] = RESULT_JSON_PATH

    output_check_rows = []

    for name, path in output_check_targets.items():
        exists = path.exists()
        rows = None
        columns = None

        if exists and path.suffix.lower() == ".csv":
            try:
                temp_df = pd.read_csv(path)
                rows = len(temp_df)
                columns = len(temp_df.columns)
            except Exception:
                rows = "읽기 실패"
                columns = "읽기 실패"

        output_check_rows.append({
            "산출물": name,
            "exists": exists,
            "rows": rows,
            "columns": columns,
            "path": str(path),
        })

    output_check = pd.DataFrame(output_check_rows)

    if show_preview:
        return output_check


if not RUN_MULTI_GAME_MODE:
    output_check = check_output_files_for_current_game(show_preview=RUN_CHECK_CELLS)
else:
    print("RUN_MULTI_GAME_MODE=True: 산출물 점검은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.")


RUN_MULTI_GAME_MODE=True: 산출물 점검은 마지막 여러 게임 실행 셀에서 게임별로 수행합니다.


# 14. 여러 게임 한 번에 실행

위 설정의 `TARGET_GAME_KEYS`에 들어 있는 게임을 순서대로 실행한다.


In [17]:
async def run_postlaunch_llm_for_game(game_key):
    """04번 전체 흐름을 게임 1개 기준으로 실행한다."""
    print("\n" + "=" * 90)
    print(f"04번 LLM 리뷰 분류 실행: {game_key}")
    print("=" * 90)

    set_target_game_context(game_key)

    global ANALYSIS_FILTERS
    ANALYSIS_FILTERS = build_analysis_filters()

    prepare_filtered_reviews(show_checks=RUN_CHECK_CELLS)
    prepare_llm_input_for_current_game(show_preview=RUN_CHECK_CELLS)
    await execute_llm_analysis_for_current_game(show_preview=RUN_CHECK_CELLS)
    postprocess_and_save_review_results(show_preview=RUN_CHECK_CELLS)
    save_issue_tags_flat_for_current_game(show_preview=RUN_CHECK_CELLS)
    output_check = check_output_files_for_current_game(show_preview=RUN_CHECK_CELLS)

    success_rows = 0
    issue_rows = 0
    if RESULT_CSV_PATH.exists():
        try:
            success_rows = len(pd.read_csv(RESULT_CSV_PATH))
        except Exception:
            success_rows = None
    if ISSUE_TAG_FLAT_PATH.exists():
        try:
            issue_rows = len(pd.read_csv(ISSUE_TAG_FLAT_PATH))
        except Exception:
            issue_rows = None

    return {
        "game_key": game_key,
        "appid": TARGET_APPID,
        "game_name": TARGET_GAME_NAME,
        "status": "success",
        "review_result_rows": success_rows,
        "issue_tag_rows": issue_rows,
        "run_dir": str(RUN_DIR),
        "result_csv_path": str(RESULT_CSV_PATH),
        "issue_tag_flat_path": str(ISSUE_TAG_FLAT_PATH),
        "error_message": "",
    }


async def run_postlaunch_llm_multi_games(game_keys):
    """04번 LLM 리뷰 분류를 여러 게임에 대해 순서대로 실행한다."""
    batch_logs = []

    for game_key in game_keys:
        try:
            log = await run_postlaunch_llm_for_game(game_key)
        except Exception as e:
            log = {
                "game_key": game_key,
                "appid": None,
                "game_name": POSTLAUNCH_TARGET_GAMES.get(game_key, {}).get("game_name", ""),
                "status": "failed",
                "review_result_rows": None,
                "issue_tag_rows": None,
                "run_dir": "",
                "result_csv_path": "",
                "issue_tag_flat_path": "",
                "error_message": str(e),
            }
            print(f"[실패] {game_key}: {e}")

        batch_logs.append(log)

    batch_log_df = pd.DataFrame(batch_logs)
    batch_log_path = RUNS_DIR / "04_postlaunch_llm_batch_log.csv"
    batch_log_df.to_csv(batch_log_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 90)
    print("04번 여러 게임 실행 요약")
    print("=" * 90)
    print("batch log 저장:", batch_log_path)
    display(batch_log_df)

    return batch_log_df


if RUN_MULTI_GAME_MODE:
    batch_log_df = await run_postlaunch_llm_multi_games(TARGET_GAME_KEYS)
else:
    print("RUN_MULTI_GAME_MODE=False: 단일 게임 실행 모드입니다.")



04번 LLM 리뷰 분류 실행: heroes_of_hammerwatch_2
최소 리뷰 길이 필터: 160,644 -> 112,192 / 제거 48,452 (30.16%)
의미 있는 리뷰 필터: 112,192 -> 53,999 / 제거 58,193 (51.87%)
appid 필터: 53,999 -> 2,681 / 제거 51,318 (95.04%)
출시일 시작 필터: 2,681 -> 2,681 / 제거 0 (0.00%)
언어 필터: 2,681 -> 2,347 / 제거 334 (12.46%)
Steam 라벨 필터: 2,347 -> 2,347 / 제거 0 (0.00%)
Steam 구매 리뷰 필터: 2,347 -> 2,347 / 제거 0 (0.00%)
무료 수령 리뷰 제외: 2,347 -> 2,343 / 제거 4 (0.17%)
얼리액세스 리뷰 제외: 2,343 -> 2,343 / 제거 0 (0.00%)
최종 필터링 리뷰 수: 2343
최종 필터링 게임 수: 1


,step,before_rows,after_rows,removed_rows,removed_rate
0,최소 리뷰 길이 필터,160644,112192,48452,0.301611
1,의미 있는 리뷰 필터,112192,53999,58193,0.518691
2,appid 필터,53999,2681,51318,0.950351
3,출시일 시작 필터,2681,2681,0,0.000000
4,언어 필터,2681,2347,334,0.124580
5,Steam 라벨 필터,2347,2347,0,0.000000
6,Steam 구매 리뷰 필터,2347,2347,0,0.000000
7,무료 수령 리뷰 제외,2347,2343,4,0.001704
8,얼리액세스 리뷰 제외,2343,2343,0,0.000000


필터링 후 release_period 분포


release_period
D0-D30      1472
D181+        435
D31-D90      305
D91-D180     131
Name: count, dtype: int64

필터링 후 Steam 라벨 분포


...

,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_primary_issue,llm_issue_tags,llm_urgency_candidate,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation,error_message
0,success,176177053,2990640,Endoparasitic 2,2024-10-01T20:00:52,2024-10-01T00:00:00,0,D0-D30,positive,0.266667,6,0.510040,positive,5,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': '한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함'}]",low,"한 손으로 플레이 가능한 조작감이 훌륭하며, 전작을 잇는 뛰어난 속편임.",현재의 직관적인 조작 방식과 게임성을 유지 및 강화할 것.,exact_match,None
1,success,176177300,2990640,Endoparasitic 2,2024-10-01T20:05:29,2024-10-01T00:00:00,0,D0-D30,positive,0.383333,1,0.490937,positive,5,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': '게임이 매우 훌륭하다고 언급함'}]",low,게임을 플레이한 후 매우 훌륭하다는 첫인상을 남김.,초반 플레이 경험이 긍정적이므로 현재의 게임 디자인 방향을 유지할 것.,exact_match,None
2,success,176179490,2990640,Endoparasitic 2,2024-10-01T20:46:59,2024-10-01T00:00:00,0,D0-D30,positive,10.433333,34,0.749335,positive,4,content_volume,"[{'category': 'content_volume', 'sentiment': 'neutral', 'evidence': '적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함'}, {'category': '...",medium,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match,None
3,success,176179851,2990640,Endoparasitic 2,2024-10-01T20:54:30,2024-10-01T00:00:00,0,D0-D30,positive,1.150000,9,0.597647,positive,5,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': 'Polished and bug free, fun additional gamepla...",low,"전작의 장점을 잘 계승한 완성도 높은 후속작으로, 버그 없이 쾌적한 플레이와 재미있는 전투 및 맵 디자인을 높게 평가함.","현재의 게임 플레이 요소와 맵 디자인의 완성도를 유지하며, 향후 업데이트에서도 안정적인 빌드 품질을 지속적으로 관리할 것.",exact_match,None
4,success,176181376,2990640,Endoparasitic 2,2024-10-01T21:27:57,2024-10-01T00:00:00,0,D0-D30,positive,2.066667,4,0.537572,positive,5,positive_praise,"[{'category': 'gameplay_loop', 'sentiment': 'positive', 'evidence': 'stuck in a corner, out of ammo and low on healt...",low,탄약 부족과 체력 고갈 등 극한의 상황에서 오는 긴장감을 긍정적으로 평가하며 게임의 재미를 극찬함.,현재의 긴장감 넘치는 게임 플레이 루프와 난이도 밸런스를 유지하여 플레이어에게 몰입감 있는 경험을 지속적으로 제공할 것.,exact_match,None


리뷰 단위 LLM 결과 JSON 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_review_analysis_result.json
리뷰 단위 LLM 결과 CSV 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_review_analysis_result.csv
성공 분석 리뷰 수: 245
실패/누락 리뷰 수: 0
전체 결과 행 수: 245
이슈 태그 펼친 결과 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_issue_tags_flat.csv
이슈 태그 행 수: 489


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence
0,176177053,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.266667,6,0.510040,positive_praise,긍정 칭찬,positive,한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함
1,176177300,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.383333,1,0.490937,positive_praise,긍정 칭찬,positive,게임이 매우 훌륭하다고 언급함
2,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,content_volume,콘텐츠 분량,neutral,적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함
3,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,gameplay_loop,게임플레이 루프,positive,크래프팅 시스템이 단순하지만 잘 작동하고 재미있음
4,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,story,스토리,positive,스토리가 전작보다 깊이 있고 재미있음



04번 여러 게임 실행 요약
batch log 저장: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\04_postlaunch_llm_batch_log.csv


,game_key,appid,game_name,status,review_result_rows,issue_tag_rows,run_dir,result_csv_path,issue_tag_flat_path,error_message
0,heroes_of_hammerwatch_2,619820,Heroes of Hammerwatch II,success,1000,1948,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_r...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\heroes_of_hammerwatch_2\llm_i...,
1,necrosmith_2,2277320,Necrosmith 2,success,324,630,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\llm_review_analy...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\necrosmith_2\llm_issue_tags_f...,
2,children_of_the_sun,1309950,Children of the Sun,success,883,1578,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\llm_revie...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\children_of_the_sun\llm_issue...,
3,laundry_store_simulator,3150440,Laundry Store Simulator,success,575,1084,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\llm_r...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\laundry_store_simulator\llm_i...,
4,endoparasitic_2,2990640,Endoparasitic 2,success,245,489,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_review_an...,c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_issue_tag...,


# 14. 출력 테이블 설명

`llm_input_reviews.csv`
- 설명: 출시 후 특정 게임 LLM 리뷰 분류에 실제로 입력된 리뷰 데이터

| 컬럼명                           | 설명                    | 예시 값                                                                       |
| ----------------------------- | --------------------- | -------------------------------------------------------------------------- |
| `recommendationid`            | Steam 리뷰 고유 ID        | 221779395                                                                  |
| `appid`                       | Steam 게임 고유 ID        | 1466060                                                                    |
| `game_name`                   | Steam 게임명             | Tainted Grail: The Fall of Avalon                                          |
| `language`                    | 리뷰 작성 언어              | english                                                                    |
| `review_datetime`             | 리뷰 작성 일시              | 2026-04-05 16:40:12                                                        |
| `release_date`                | 게임 출시일                | 2025-05-23                                                                 |
| `days_from_release`           | 출시일 기준 리뷰 작성일까지 지난 일수 | 317                                                                        |
| `release_period`              | 출시 후 리뷰 작성 구간         | post_D30                                                                   |
| `release_period_detail`       | 출시 후 세부 리뷰 작성 구간      | older                                                                      |
| `steam_label_text`            | Steam 추천/비추천 라벨       | negative                                                                   |
| `voted_up`                    | Steam 원본 추천 여부        | False                                                                      |
| `playtime_at_review_hours`    | 리뷰 작성 시점 플레이타임        | 12.5                                                                       |
| `playtime_stage`              | 리뷰 작성 시점 플레이타임 구간     | 5-20h                                                                      |
| `votes_up`                    | 리뷰가 받은 유용함 투표 수       | 4                                                                          |
| `weighted_vote_score`         | Steam 리뷰 가중 점수        | 0.612                                                                      |
| `received_for_free`           | 무료 수령 여부              | False                                                                      |
| `written_during_early_access` | 얼리액세스 기간 작성 여부        | False                                                                      |
| `price`                       | 게임 가격                 | 29.99                                                                      |
| `price_group`                 | 게임 가격 구간              | 20-30                                                                      |
| `genres_text`                 | 게임 장르 목록              | Action, Adventure, Indie, RPG                                              |
| `categories_text`             | Steam 카테고리 목록         | Single-player, Steam Achievements                                          |
| `top_steam_tags_text`         | 주요 Steam 태그 목록        | RPG, Open World, Dark Fantasy                                              |
| `review_text_for_llm`         | LLM에 실제로 전달한 리뷰 본문    | The game is fun, but performance issues and bugs make it hard to continue. |
| `is_meaningful_review`        | 의미 있는 리뷰 여부           | True                                                                       |
| `meaningless_reason`          | 무의미한 리뷰로 판단된 이유       | meaningful                                                                 |



`llm_input_filter_log.csv`
- 설명: 출시 후 LLM 입력 리뷰를 만들기 전 필터링 단계별 행 수 변화 기록

| 컬럼명            | 설명              | 예시 값     |
| -------------- | --------------- | -------- |
| `step`         | 적용한 필터링 단계명     | 대상 게임 필터 |
| `before_rows`  | 해당 필터 적용 전 행 수  | 236379   |
| `after_rows`   | 해당 필터 적용 후 행 수  | 11982    |
| `removed_rows` | 해당 필터에서 제거된 행 수 | 224397   |
| `removed_rate` | 제거 비율           | 0.9493   |


`llm_input_sample_summary.csv`
- 설명: 출시 후 LLM 입력으로 샘플링된 게임별 리뷰 수 요약

| 컬럼명                     | 설명                   | 예시 값                              |
| ----------------------- | -------------------- | --------------------------------- |
| `appid`                 | Steam 게임 고유 ID       | 1466060                           |
| `game_name`             | Steam 게임명            | Tainted Grail: The Fall of Avalon |
| `sampled_review_count`  | 최종 LLM 입력으로 선택된 리뷰 수 | 300                               |
| `positive_count`        | 샘플 내 Steam 긍정 리뷰 수   | 150                               |
| `negative_count`        | 샘플 내 Steam 부정 리뷰 수   | 150                               |
| `first_review_datetime` | 샘플 내 가장 이른 리뷰 작성 시점  | 2025-05-23 10:20:11               |
| `last_review_datetime`  | 샘플 내 가장 최근 리뷰 작성 시점  | 2026-04-05 16:40:12               |



`llm_review_analysis_result.csv`
- 설명: 출시 후 리뷰 1개 단위 LLM 감정·이슈 분류 결과

| 컬럼명                            | 설명                                     | 예시 값                                                |
| ------------------------------ | -------------------------------------- | --------------------------------------------------- |
| `analysis_status`              | LLM 분석 성공/실패 여부                        | success                                             |
| `recommendationid`             | Steam 리뷰 고유 ID                         | 221779395                                           |
| `appid`                        | Steam 게임 고유 ID                         | 1466060                                             |
| `game_name`                    | Steam 게임명                              | Tainted Grail: The Fall of Avalon                   |
| `review_datetime`              | 리뷰 작성 일시                               | 2026-04-05 16:40:12                                 |
| `release_date`                 | 게임 출시일                                 | 2025-05-23                                          |
| `days_from_release`            | 출시일 기준 리뷰 작성일까지 지난 일수                  | 317                                                 |
| `release_period`               | 출시 후 리뷰 작성 구간                          | post_D30                                            |
| `steam_label_text`             | Steam 추천/비추천 라벨                        | negative                                            |
| `playtime_at_review_hours`     | 리뷰 작성 시점 플레이타임                         | 12.5                                                |
| `votes_up`                     | 리뷰가 받은 유용함 투표 수                        | 4                                                   |
| `weighted_vote_score`          | Steam 리뷰 가중 점수                         | 0.612                                               |
| `llm_sentiment`                | LLM이 판단한 리뷰 전체 감정                      | mixed                                               |
| `sentiment_score`              | LLM이 판단한 감정 점수                         | 2                                                   |
| `llm_primary_issue`            | 리뷰의 대표 이슈                              | performance                                         |
| `llm_issue_tags`               | 리뷰 안에서 발견된 세부 이슈 태그 목록                 | [{"category":"performance","sentiment":"negative"}] |
| `llm_urgency_candidate`        | LLM이 리뷰 문맥을 보고 분류한 시급도 후보. 최종 우선순위는 아님 | high                                                |
| `llm_review_summary`           | LLM이 요약한 리뷰 핵심 내용                      | 전반적인 재미는 있지만 성능 문제와 버그로 플레이 경험이 저하됨                 |
| `llm_suggested_action`         | LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보          | 성능 저하와 진행 방해 버그를 우선 확인                              |
| `steam_llm_sentiment_relation` | Steam 라벨과 LLM 감정 판단의 관계                | partial_match                                       |


`llm_issue_tags_flat.csv`
- 설명: 출시 후 리뷰의 LLM 이슈 태그를 1행 1이슈 형태로 펼친 데이터

| 컬럼명                        | 설명                        | 예시 값                                                 |
| -------------------------- | ------------------------- | ---------------------------------------------------- |
| `recommendationid`         | Steam 리뷰 고유 ID            | 221779395                                            |
| `appid`                    | Steam 게임 고유 ID            | 1466060                                              |
| `game_name`                | Steam 게임명                 | Tainted Grail: The Fall of Avalon                    |
| `steam_label_text`         | Steam 추천/비추천 라벨           | negative                                             |
| `llm_sentiment`            | LLM이 판단한 리뷰 전체 감정         | mixed                                                |
| `llm_primary_issue`        | 리뷰의 대표 이슈                 | performance                                          |
| `llm_urgency_candidate`    | LLM이 리뷰 문맥을 보고 분류한 시급도 후보 | high                                                 |
| `release_period`           | 출시 후 리뷰 작성 구간             | post_D30                                             |
| `playtime_at_review_hours` | 리뷰 작성 시점 플레이타임            | 12.5                                                 |
| `votes_up`                 | 리뷰가 받은 유용함 투표 수           | 4                                                    |
| `weighted_vote_score`      | Steam 리뷰 가중 점수            | 0.612                                                |
| `llm_issue_category`       | 세부 이슈 카테고리                | performance                                          |
| `issue_name_kor`           | 세부 이슈의 한국어 이름             | 성능                                                   |
| `llm_issue_sentiment`      | LLM이 세부 이슈 단위로 분류한 감정 방향  | negative                                             |
| `llm_issue_evidence`       | LLM이 해당 태그를 판단한 근거 문장     | performance issues and bugs make it hard to continue |



